In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 1. DB 연결
db_path = 'chembl_36/chembl_36_sqlite/chembl_36.db' 
conn = sqlite3.connect(db_path)

# =========================================================
# [Table 1] 메인 분자 정보 (1분자 = 1행)
# =========================================================
# 여기에는 중복을 일으키는 Synonyms를 제외하고, 고유한 정보만 넣습니다.
query_main = """
    SELECT
        md.chembl_id AS chembl_id,
        md.pref_name AS name,
        md.max_phase AS max_phase,
        md.first_approval AS first_approval,
        cs.canonical_smiles AS smiles,
        cs.standard_inchi_key AS inchi_key
    FROM
        MOLECULE_DICTIONARY md
    LEFT JOIN
        COMPOUND_STRUCTURES cs ON md.molregno = cs.molregno
"""

print("1. 메인 분자 정보 추출 중...")
df_main = pd.read_sql_query(query_main, conn)

# =========================================================
# [Table 2] 이명(Synonyms) 매핑 테이블 (1분자 = N행)
# =========================================================
# ChEMBL_ID와 Synonym만 따로 뽑습니다. 
# INNER JOIN을 써서 별명이 있는 경우만 가져옵니다.
# [Table 2] 이명(Synonyms) 매핑 테이블 (안전 장치 추가)
query_synonyms = """
    SELECT
        md.chembl_id AS chembl_id,
        ms.synonyms AS synonym,
        ms.syn_type AS synonym_type
    FROM
        MOLECULE_DICTIONARY md
    JOIN
        MOLECULE_SYNONYMS ms ON md.molregno = ms.molregno
    WHERE
        md.chembl_id IS NOT NULL  -- [핵심] ID가 없는 유령 데이터는 제외!
"""

print("2. 이명(Synonyms) 정보 추출 중...")
df_synonyms = pd.read_sql_query(query_synonyms, conn)

conn.close()

# =========================================================
# [데이터 전처리] NaN -> None 변환
# =========================================================
df_main = df_main.replace({np.nan: None})
df_synonyms = df_synonyms.replace({np.nan: None})

# =========================================================
# [결과 확인]
# =========================================================
print("\n" + "="*30)
print(f"[메인 테이블] 크기: {df_main.shape}")
print("="*30)
print(df_main.head())

print("\n" + "="*30)
print(f"[이명 테이블] 크기: {df_synonyms.shape}")
print("="*30)
# 특정 약물(예: Viagra 성분)의 이명이 어떻게 저장되었는지 예시로 확인
sample_id = df_synonyms['chembl_id'].iloc[0] # 첫 번째 ID 샘플
print(f"예시 ({sample_id})의 이명 목록:")
print(df_synonyms[df_synonyms['chembl_id'] == sample_id].head(5))

# =========================================================
# [저장 예시]
# =========================================================
# df_main.to_csv('chembl_molecules.csv', index=False)
# df_synonyms.to_csv('chembl_synonyms.csv', index=False)

1. 메인 분자 정보 추출 중...
2. 이명(Synonyms) 정보 추출 중...

[메인 테이블] 크기: (2878135, 6)
      chembl_id  name max_phase first_approval  \
0    CHEMBL6329  None      None           None   
1    CHEMBL6328  None      None           None   
2  CHEMBL265667  None      None           None   
3    CHEMBL6362  None      None           None   
4  CHEMBL267864  None      None           None   

                                             smiles  \
0      Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl   
1   Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1   
2  Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1   
3      Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1   
4    Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1   

                     inchi_key  
0  OWRSAHYFSSNENM-UHFFFAOYSA-N  
1  ZJYUMURGSZQFMH-UHFFFAOYSA-N  
2  YOMWDCALSDWFSV-UHFFFAOYSA-N  
3  PSOPUAQFGCRDIP-UHFFFAOYSA-N  
4  KEZNSCMBVRNOHO-UHFFFAOYSA-N  

[이명 테이블] 크기: (132937, 3)
예시 (CHEMBL6461)의 이명 목록:
    chembl_id    synonym   synonym_type
0  CHE